In [1]:
!pip install Faker

  Using cached faker-40.11.1-py3-none-any.whl.metadata (16 kB)
Using cached faker-40.11.1-py3-none-any.whl (2.0 MB)


In [1]:
import sqlite3
import random
import os
from datetime import datetime, timedelta
from faker import Faker
import calendar

In [2]:
fake = Faker("en_IE")
random.seed(42)
Faker.seed(42)
 
NUM_CUSTOMERS      = 50_000
BILLING_CYCLES_PER = 10        # ~10 billing cycles per customer → ~500k rows
DB_PATH            = "data/legacy.db"
os.makedirs("data", exist_ok=True)
os.makedirs("data/exports", exist_ok=True)
 
TARIFF_PLANS  = ["Standard", "Economy 7", "Smart Saver", "Business Rate", "Green Tariff"]
FUEL_TYPES    = ["Gas", "Electricity", "Dual Fuel"]
REGIONS       = ["Dublin", "Cork", "Galway", "Limerick", "Waterford", "Kilkenny", "Sligo"]
PAYMENT_METHODS = ["Direct Debit", "Credit Card", "Bank Transfer", "Prepay"]

In [3]:
def random_date(start: datetime, end: datetime) -> str:
    delta = end - start
    return (start + timedelta(days=random.randint(0, delta.days))).strftime("%d/%m/%Y")  # Legacy format: DD/MM/YYYY
 
def next_month(date_str: str) -> str:
    d = datetime.strptime(date_str, "%d/%m/%Y")
    if d.month == 12:
        next_month = 1
        year = d.year + 1
    else:
        next_month = d.month + 1
        year = d.year
    # Get the last day of the target month
    last_day = calendar.monthrange(year, next_month)[1]
    # Set the day to the minimum of the original day and the last day
    new_day = min(d.day, last_day)
    new_date = d.replace(year=year, month=next_month, day=new_day)
    return new_date.strftime("%d/%m/%Y")

In [4]:
def create_schema(conn: sqlite3.Connection):
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS customers (
            customer_id     TEXT PRIMARY KEY,
            first_name      TEXT NOT NULL,
            last_name       TEXT NOT NULL,
            email           TEXT,
            phone           TEXT,
            address         TEXT,
            region          TEXT,
            fuel_type       TEXT,
            tariff_plan     TEXT,
            account_open_date TEXT,
            payment_method  TEXT,
            is_active       INTEGER DEFAULT 1
        );
 
        CREATE TABLE IF NOT EXISTS accounts (
            account_id      TEXT PRIMARY KEY,
            customer_id     TEXT NOT NULL,
            account_type    TEXT,
            credit_limit    REAL,
            current_balance REAL,
            last_payment_date TEXT,
            FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
        );
 
        CREATE TABLE IF NOT EXISTS billing_cycles (
            cycle_id        TEXT PRIMARY KEY,
            account_id      TEXT NOT NULL,
            cycle_start     TEXT,
            cycle_end       TEXT,
            units_consumed  REAL,
            unit_rate       REAL,
            standing_charge REAL,
            vat_rate        REAL,
            amount_due      REAL,
            due_date        TEXT,
            status          TEXT,
            FOREIGN KEY (account_id) REFERENCES accounts(account_id)
        );
 
        CREATE TABLE IF NOT EXISTS payments (
            payment_id      TEXT PRIMARY KEY,
            account_id      TEXT NOT NULL,
            cycle_id        TEXT,
            payment_date    TEXT,
            amount_paid     REAL,
            payment_method  TEXT,
            status          TEXT,
            FOREIGN KEY (account_id) REFERENCES accounts(account_id),
            FOREIGN KEY (cycle_id) REFERENCES billing_cycles(cycle_id)
        );
    """)
    conn.commit()

In [5]:
def populate(conn: sqlite3.Connection):
    customers, accounts, cycles, payments = [], [], [], []
 
    start_range = datetime(2020, 1, 1)
    end_range   = datetime(2025, 1, 1)
 
    print(f"Generating {NUM_CUSTOMERS:,} customers...")
 
    for i in range(NUM_CUSTOMERS):
        cust_id    = f"CUST{i+1:06d}"
        acct_id    = f"ACCT{i+1:06d}"
        fuel       = random.choice(FUEL_TYPES)
        tariff     = random.choice(TARIFF_PLANS)
        region     = random.choice(REGIONS)
        pay_method = random.choice(PAYMENT_METHODS)
        open_date  = random_date(start_range, end_range)
 
        customers.append((
            cust_id, fake.first_name(), fake.last_name(),
            fake.email(), fake.phone_number(),
            fake.street_address(), region, fuel, tariff,
            open_date, pay_method,
            1 if random.random() > 0.05 else 0   # 5% inactive
        ))
 
        current_balance = round(random.uniform(-500, 2000), 2)
        last_pay_date   = random_date(datetime(2023, 1, 1), datetime(2024, 6, 1))
        accounts.append((
            acct_id, cust_id,
            "Residential" if fuel != "Business Rate" else "Commercial",
            round(random.uniform(1000, 5000), 2),
            current_balance, last_pay_date
        ))
 
        # Billing cycles
        cycle_start = open_date
        for j in range(BILLING_CYCLES_PER):
            cycle_id      = f"CYC{i+1:06d}{j+1:03d}"
            cycle_end     = next_month(cycle_start)
            units         = round(random.uniform(50, 800), 2)
            unit_rate     = round(random.uniform(0.18, 0.45), 4)
            standing      = round(random.uniform(15, 45), 2)
            vat           = 0.135                                    # Irish VAT on energy
            net_amount    = round(units * unit_rate + standing, 2)
            amount_due    = round(net_amount * (1 + vat), 2)
            due_date      = next_month(cycle_end)
            status        = random.choices(
                ["Paid", "Unpaid", "Partially Paid", "Overdue"],
                weights=[70, 10, 10, 10]
            )[0]
 
            cycles.append((
                cycle_id, acct_id, cycle_start, cycle_end,
                units, unit_rate, standing, vat,
                amount_due, due_date, status
            ))
 
            # Payment record if not unpaid
            if status != "Unpaid":
                paid = amount_due if status == "Paid" else round(amount_due * random.uniform(0.3, 0.9), 2)
                pay_id = f"PAY{i+1:06d}{j+1:03d}"
                pay_date = next_month(cycle_start)
                payments.append((
                    pay_id, acct_id, cycle_id, pay_date,
                    paid, pay_method,
                    "Completed" if status == "Paid" else "Partial"
                ))
 
            cycle_start = cycle_end
 
        if (i + 1) % 10_000 == 0:
            print(f"  {i+1:,} customers generated...")
 
    print("Writing to database...")
    conn.executemany("INSERT INTO customers VALUES (?,?,?,?,?,?,?,?,?,?,?,?)", customers)
    conn.executemany("INSERT INTO accounts VALUES (?,?,?,?,?,?)", accounts)
 
    # Insert cycles in batches (large volume)
    batch = 10_000
    for k in range(0, len(cycles), batch):
        conn.executemany("INSERT INTO billing_cycles VALUES (?,?,?,?,?,?,?,?,?,?,?)", cycles[k:k+batch])
 
    for k in range(0, len(payments), batch):
        conn.executemany("INSERT INTO payments VALUES (?,?,?,?,?,?,?)", payments[k:k+batch])
 
    conn.commit()
    print(f"\n✅ Legacy DB ready at {DB_PATH}")
    print(f"   Customers:      {len(customers):,}")
    print(f"   Accounts:       {len(accounts):,}")
    print(f"   Billing Cycles: {len(cycles):,}")
    print(f"   Payments:       {len(payments):,}")

In [6]:
if __name__ == "__main__":
    conn = sqlite3.connect(DB_PATH)
    create_schema(conn)
    populate(conn)
    conn.close()

Generating 50,000 customers...
  10,000 customers generated...
  20,000 customers generated...
  30,000 customers generated...
  40,000 customers generated...
  50,000 customers generated...
Writing to database...

✅ Legacy DB ready at data/legacy.db
   Customers:      50,000
   Accounts:       50,000
   Billing Cycles: 500,000
   Payments:       449,773
